# afivo-streamer を Python から実行する

このノートブックでは次の流れを1か所で行います。

1. 元になる `.cfg` を選び、変えたい設定を Python の辞書で指定する
2. `streamer` を実行する（設定はコマンドライン引数 `-名前=値` で上書き）
3. ログファイルからストリーマーの速度・電界などをグラフにする
4. `.silo` ファイルから分布図・動画を作る
5. パラメータを変えて複数回計算し、比較する

**前提**：`~/afivo-streamer` で `make` 済みで、`afivo/tools` で `make`（`silo_to_raw`）済みであること。

## 1. 準備

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib_fontja  # 日本語フォント
import numpy as np
import pandas as pd
from IPython.display import Image, display

# リポジトリの場所（このノートブックは notebooks/ にある想定）
REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.home() / "afivo-streamer"
TOOLS = REPO / "afivo" / "tools"
print("REPO:", REPO)
assert (TOOLS / "silo_to_raw").exists(), "afivo/tools で make を実行してください"

In [ ]:
def _fmt(value):
    """Python の値を afivo-streamer の設定値の文字列に変換する"""
    if isinstance(value, bool):
        return "T" if value else "F"
    if isinstance(value, (list, tuple)):
        return " ".join(_fmt(v) for v in value)
    return str(value)


def run_streamer(program, cfg, name, overrides=None, threads=4, show_every=1):
    """streamer を実行する。

    program   : programs/ 以下のフォルダ名（例 "standard_2d"）
    cfg       : 元にする設定ファイル名（例 "streamer_cyl.cfg"）
    name      : 計算の名前。結果は programs/<program>/output/<name>_* に出る
    overrides : 上書きする設定 {名前: 値}
    threads   : OpenMP スレッド数（このPCでは 4 程度が速い）
    """
    workdir = REPO / "programs" / program
    (workdir / "output").mkdir(exist_ok=True)
    overrides = dict(overrides or {})
    overrides["output%name"] = f"output/{name}"

    cmd = ["./streamer", cfg] + [f"-{k}={_fmt(v)}" for k, v in overrides.items()]
    print("実行:", " ".join(cmd))
    env = dict(os.environ, OMP_NUM_THREADS=str(threads))

    t0 = time.time()
    logfile = workdir / "output" / f"{name}_stdout.txt"
    with open(logfile, "w") as log, subprocess.Popen(
            cmd, cwd=workdir, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True) as proc:
        n = 0
        for line in proc.stdout:
            log.write(line)
            if "%" in line and "it:" in line:      # 進捗の行
                n += 1
                if n % show_every == 0:
                    print(f"  [{time.time()-t0:6.0f} s] {line.strip()}")
    if proc.returncode != 0:
        print(open(logfile).read()[-3000:])
        raise RuntimeError(f"streamer が異常終了しました (code {proc.returncode})")
    print(f"完了: {time.time()-t0:.0f} 秒  (標準出力: {logfile})")
    return workdir / "output" / name


def load_log(base):
    """<name>_log.txt を DataFrame として読む（列: time, v, max(E), ...）"""
    df = pd.read_csv(f"{base}_log.txt", sep=r"\s+")
    return df


def silo_files(base):
    return sorted(Path(base).parent.glob(Path(base).name + "_[0-9]*.silo"))


def load_silo(silo, variable, r_max=None):
    """silo ファイルの1変数を一様格子に変換して (data, r, z, time) を返す。data の形は (n_r, n_z)"""
    npz = Path("/tmp") / f"_nb_{os.getpid()}.npz"
    cmd = [sys.executable, str(TOOLS / "plot_raw_data.py"), str(silo),
           "-variable", variable, "-save_npz", str(npz), "-q"]
    if r_max is not None:
        cmd += ["-r_max", *map(str, r_max)]
    subprocess.run(cmd, check=True)
    d = np.load(npz)
    return d["uniform_data"], d["arr_0"], d["arr_1"], float(d["time"])

## 2. 設定を決めて計算する

`streamer_cyl.cfg`（軸対称、空気、背景電界 2 MV/m）を元に、終了時刻だけ短くして実行します。
`overrides` に書いたものだけが上書きされ、それ以外は `.cfg` の値が使われます。

使える設定名の一覧は、一度計算すると `output/<name>_out.cfg` に説明付きで出力されます。

In [ ]:
base = run_streamer(
    program="standard_2d",
    cfg="streamer_cyl.cfg",
    name="nb_cyl",
    overrides={
        "end_time": 5e-9,            # 終了時刻 (s)
        "field_amplitude": -2.0e6,   # 背景電界 (V/m)、負で下向き
        "output%dt": 0.25e-9,        # 出力間隔 (s)
    },
)

## 3. ログファイルを見る

In [ ]:
df = load_log(base)
df[["it", "time", "v", "max(E)", "ne_zmin", "n_cells"]].tail()

In [ ]:
t_ns = df["time"] * 1e9
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
ax[0].plot(t_ns, df["max(E)"] / 1e6)
ax[0].set_ylabel("max |E| (MV/m)")
ax[1].plot(t_ns[1:], df["ne_zmin"][1:] * 1e3)  # t=0 は先端が未定義なので除く
ax[1].set_ylabel("先端位置 ne_zmin (mm)")
ax[2].plot(t_ns, df["v"] / 1e3)
ax[2].set_ylabel("速度 v (km/s)")
for a in ax:
    a.set_xlabel("t (ns)")
    a.grid(alpha=0.3)

## 4. 分布図と動画

In [ ]:
files = silo_files(base)
print(len(files), "個の silo ファイル")

fig, axes = plt.subplots(1, 2, figsize=(8, 5), constrained_layout=True)
for ax, var, label in zip(axes, ["e", "electric_fld"],
                          ["電子密度 (m$^{-3}$)", "電界 (V/m)"]):
    data, r, z, t = load_silo(files[-1], var, r_max=[4e-3, 16e-3])
    im = ax.imshow(data.T, origin="lower", extent=[r[0]*1e3, r[-1]*1e3, z[0]*1e3, z[-1]*1e3],
                   aspect="equal", cmap="inferno" if var == "e" else "viridis")
    ax.set_xlabel("r (mm)"); ax.set_ylabel("z (mm)")
    fig.colorbar(im, ax=ax).set_label(label)
fig.suptitle(f"t = {t*1e9:.2f} ns")

In [ ]:
gif = base.parent.parent / "plots" / f"{base.name}.gif"
gif.parent.mkdir(exist_ok=True)
subprocess.run([sys.executable, str(REPO / "tools" / "make_movie.py"), str(base),
                "-o", str(gif), "-z_max", "0.012"], check=True)
display(Image(filename=str(gif)))

## 5. パラメータスキャン

背景電界を変えて計算し、先端位置の時間変化を比べます。
1回あたり数十秒かかります。

In [ ]:
fields = [-1.5e6, -2.0e6, -2.5e6]   # V/m
results = {}
for E0 in fields:
    name = f"nb_scan_E{abs(E0)/1e6:.1f}MV"
    b = run_streamer("standard_2d", "streamer_cyl.cfg", name,
                     overrides={"end_time": 4e-9, "field_amplitude": E0},
                     show_every=10**6)  # 進捗表示なし
    results[E0] = load_log(b)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for E0, d in results.items():
    lbl = f"{abs(E0)/1e6:.1f} MV/m"
    ax[0].plot(d["time"][1:] * 1e9, d["ne_zmin"][1:] * 1e3, label=lbl)
    ax[1].plot(d["time"] * 1e9, d["max(E)"] / 1e6, label=lbl)
ax[0].set_ylabel("先端位置 ne_zmin (mm)")
ax[1].set_ylabel("max |E| (MV/m)")
for a in ax:
    a.set_xlabel("t (ns)"); a.grid(alpha=0.3); a.legend()